# Calculate a high-resolution BeamFactor

This notebook calculates a high-resolution beam factor that can be interpolated and applied to any particular day in a fast way. The defaults are set in such a way as to reproduce Alan's beam factor, and to show this, we give an example of a couple of days in this notebook.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from edges_analysis import beams, sky_models
from astropy import units as un
import numpy as np
from edges_analysis import const
from edges_cal import modelling as mdl
import hickle

In [ ]:
outdir: str = "."
beamfile: str = '/data4/nmahesh/edges/code_from_alan_org/newniv.txt'  # probably should move this into the repo...

alandata: str = "/data4/smurray/edges/alans-pipeline/scripts/H2CaseFieldData/"

f_low: float = 40.0
f_high: float = 100.0
rotation_from_north: float = -6
lst_start: float = 0.0
dlst: float = 0.1
sky_model: str = "Haslam408AllNoh"
index_model: str = "ConstantIndex"
normalize_beam: bool = False
beam_smoothing: bool = False
interp_kind: str = 'nearest'
telescope = 'edges-low-alan'
sky_at_reference_frequency: bool = False
use_astropy_azel: bool = False

In [ ]:
beam = beams.Beam.from_file(band='low', beam_file=beamfile, rotation_from_north=rotation_from_north)

In [ ]:
lsts = np.arange(lst_start, lst_start + 24.0, dlst)

In [ ]:
beamfac = beams.antenna_beam_factor(
    beam,
    f_low=f_low*un.MHz, 
    f_high=f_high*un.MHz, 
    lsts=lsts, 
    sky_model=getattr(sky_models, sky_model)(), 
    index_model= getattr(sky_models, index_model)(),
    normalize_beam=normalize_beam,
    ground_loss_file=None,
    reference_frequency=75*un.MHz,
    beam_smoothing=beam_smoothing,
    interp_kind=interp_kind,
    freq_progress=False,
    location=const.KNOWN_TELESCOPES[telescope].location,
    sky_at_reference_frequency=sky_at_reference_frequency,
    use_astropy_azel=use_astropy_azel
)

In [ ]:
h2casedir = Path(alandata)
dirs = [d for d in sorted(h2casedir.glob("???")) if d.is_dir()][:10]

In [ ]:
fourier = mdl.Fourier(
    n_terms=31,
    transform=mdl.ShiftTransform(shift=75.0),
    period=1.2*beamfac.nfreq * (beamfac.frequencies[1] - beamfac.frequencies[0]),
)

In [ ]:

fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 9),constrained_layout=True)

for day in dirs[:1]:
    if not (day / 'beamcorr.txt').exists():
        continue
    
    alandata = np.genfromtxt(day / 'beamcorr.txt')
    
    beamfacs = sorted(day.glob("beamfac*.txt"), key=lambda pth: int(pth.stem[7:]))
    
    # Get all lsts from outputs
    these_lsts = []
    for bf in beamfacs:
        with open(bf, 'r') as fl:
            these_lsts.append(float(fl.readlines()[1].split("=")[-1].strip()))
            
    these_lsts = np.unwrap(np.array(these_lsts), period=24.0)
    trunc = beamfac.at_lsts(np.array(these_lsts)).get_integrated_beam_factor(model=fourier, freqs=alandata[:, 0])
    ax[0].plot(alandata[:, 0], trunc, label=str(day.name))
    ax[1].plot(alandata[:, 0], (trunc*alandata[:,3] - 1))
    
ax[0].set_ylabel("Beam Factor")
ax[1].set_ylabel("Fractional Difference")
ax[1].set_xlabel("Frequency [MHz]")
ax[0].legend()

In [ ]:
hickle.dump(beamfac, Path(outdir) / "beam_factor.hickle")